# NETRA — run the heavy processing on a free GPUThis notebook runs **the same four commands** the laptop runs — `preflight`,`run`, `resolve`, `analytics` — on whatever machine this notebook is executing on,then packs the results into one archive you download and import locally.It works on **Google Colab** and on **Kaggle Notebooks**. Both give a free GPUwith no card and no key. Nothing here is Colab-specific; if you have a GPU boxover SSH instead, skip the notebook entirely and run:```bash backend/scripts/remote_setup.shpython backend/scripts/remote_batch.py --workers 4 --max-frames 0```**What does NOT change:** the project structure, the database schema, the API,the dashboard, the event bus, or any pipeline code. Processing already wrote to`data/netra.db` and `data/evidence/` while serving only read them, so moving theprocessing elsewhere needs no architectural change — only a way to carry theresults back, which is what `export` / `import` are for.> **Before you start:** turn the GPU on.> Colab — *Runtime → Change runtime type → T4 GPU*.> Kaggle — *Settings → Accelerator → GPU*.> A CPU runtime will still work; it will just be no faster than your laptop.

## 1. What machine did we actually get?Free tiers do not always hand out a GPU. Check before spending an hour.

In [ ]:
!nvidia-smi || echo "NO GPU on this runtime -- this will be a CPU run."

## 2. Get the project onto this machinePick **one** of the three cells below and run it.The zip is the reliable option: on your laptop run `make bundle-src` (or`zip -r netra.zip . -x '*/.venv/*' '*/node_modules/*' '*/data/netra.db'`)and upload the result.

In [ ]:
# OPTION A -- upload a zip of the project (works on Colab and Kaggle)import os, sys, zipfile, pathlibif os.path.isdir("/kaggle"):    print("Kaggle: add your project as a Dataset, then set SRC to its path, e.g.")    print("  SRC = '/kaggle/input/netra/netra.zip'")    SRC = "/kaggle/input/netra/netra.zip"else:    from google.colab import files          # noqa    up = files.upload()    SRC = list(up)[0]with zipfile.ZipFile(SRC) as z:    z.extractall("/content/netra" if not os.path.isdir("/kaggle") else "/kaggle/working/netra")ROOT = "/content/netra" if not os.path.isdir("/kaggle") else "/kaggle/working/netra"# the zip may contain one top-level folder; find the real rootfor p in pathlib.Path(ROOT).rglob("configs/system.yaml"):    ROOT = str(p.parent.parent); breakos.environ["NETRA_ROOT"] = ROOTos.chdir(ROOT)print("project root:", ROOT)print(sorted(os.listdir(ROOT)))

In [ ]:
# OPTION B -- clone from git, if you have pushed the project somewhere# !git clone https://github.com/<you>/<repo>.git /content/netra# import os; os.environ["NETRA_ROOT"] = "/content/netra"; os.chdir("/content/netra")

In [ ]:
# OPTION C (Colab only) -- keep the project in Google Drive so results survive# a disconnect and you never re-upload footage.# from google.colab import drive; drive.mount('/content/drive')# import os; ROOT='/content/drive/MyDrive/netra'# os.environ["NETRA_ROOT"]=ROOT; os.chdir(ROOT)

## 3. Install dependencies`remote_setup.sh` picks the right ONNX Runtime for this machine. That matters:the CPU wheel and the GPU wheel are **different packages with the same importname**, so installing the wrong one leaves ANPR — 31% of the runtime — silentlyon the CPU while everything appears to work.

In [ ]:
!bash backend/scripts/remote_setup.sh

## 4. Upload the footageVideos are not in the project zip (too big). Put them where `configs/cameras.yaml`expects — by default `data/videos/`, or `data/test/...` for fetched footage.On Kaggle, attach the videos as a Dataset and symlink the folder instead of uploading.

In [ ]:
import os, shutilos.makedirs("data/videos", exist_ok=True)if os.path.isdir("/kaggle"):    # e.g. !ln -sfn /kaggle/input/netra-footage/videos data/videos    print("Kaggle: attach a Dataset with your clips and symlink it into data/videos")else:    from google.colab import files          # noqa    for name in files.upload():        shutil.move(name, f"data/videos/{name}")print(sorted(os.listdir("data/videos")))

## 5. Check the footage before spending GPU time on it`preflight` opens every configured source and says whether it can produceanything: **ready**, **placeholder** (a valid file containing no footage —the camera was offline), **corrupt**, **unsupported** or **missing**.Run it first. An hour of GPU time spent on offline placeholders produces aclean run, zero sightings, and no explanation.

In [ ]:
!python -m app.cli preflight --json data/exports/preflight.json

## 6. Run the batch`--max-frames 0` processes the whole clip. Start smaller (`--max-frames 150`)to confirm everything works before committing to a long run.`--workers` is how many cameras run at once. On one GPU, **2–4** is usuallyright whatever the core count: each worker holds its own copy of YOLO and twoONNX sessions, so VRAM is the binding constraint, not compute.

In [ ]:
%env PYTHONPATH=backend!python backend/scripts/remote_batch.py --workers 4 --max-frames 0

## 7. Download the resultsOne archive: the database, every evidence snapshot, the configs that producedthem, and a manifest recording which machine ran it.

In [ ]:
import glob, osbundles = sorted(glob.glob("data/exports/*.tar.gz"), key=os.path.getmtime)latest = bundles[-1]print(latest, f"{os.path.getsize(latest)/1e6:.1f} MB")if os.path.isdir("/kaggle"):    import shutil; shutil.copy(latest, "/kaggle/working/")    print("saved to /kaggle/working -- download it from the Output panel")else:    from google.colab import files          # noqa    files.download(latest)

## 8. Back on your laptop```python -m app.cli import netra-<timestamp>.tar.gzmake api     # terminal 1make dash    # terminal 2````import` **merges**. A run already in your database is skipped rather thanduplicated, so importing the same bundle twice is safe. Your existing resultsare left alone. Use `--dry-run` first if you want to see what it would do.